# ⚽ Pass Network & Team Structure Analysis
**Julie Landrevie** — Football Data & Video Analyst  
**Source** : StatsBomb Open Data  
**Stack** : Python · mplsoccer · NetworkX · pandas

---
Ce notebook couvre :
1. 📦 Import et chargement des données StatsBomb
2. 🕸️ Construction du réseau de passes
3. 📊 Métriques de centralité (NetworkX)
4. 🌡️ Heatmap de positions moyennes
5. ⚔️ Comparaison entre deux équipes
6. 💾 Export des résultats


## 1. 📦 Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgba
import networkx as nx
from mplsoccer import Pitch, VerticalPitch
from statsbombpy import sb
import warnings
warnings.filterwarnings("ignore")

# ── Style matplotlib dark
plt.rcParams.update({
    'figure.facecolor': '#0a0e1a',
    'axes.facecolor': '#0a0e1a',
    'axes.edgecolor': '#2d3748',
    'axes.labelcolor': '#a0aec0',
    'xtick.color': '#718096',
    'ytick.color': '#718096',
    'text.color': '#e8eaf0',
    'grid.color': '#2d3748',
    'font.family': 'monospace',
})

DARK_BG   = '#0a0e1a'
PITCH_CLR = '#2d3748'
ACCENT    = '#4fd1c5'
ACCENT2   = '#f6ad55'
print("✅ Imports OK")


## 2. 📥 Chargement des données StatsBomb

In [ ]:
# ── Compétitions disponibles
comps = sb.competitions()
comps[['competition_id','competition_name','season_id','season_name']].head(20)


In [ ]:
# ── Sélection : La Liga 2020/2021
COMPETITION_ID = 11
SEASON_ID      = 90

matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
matches[['match_id','home_team','away_team','home_score','away_score']].head(10)


In [ ]:
# ── Chargement d'un match spécifique
# Changez MATCH_ID pour analyser un autre match
MATCH_ID = matches['match_id'].iloc[0]
match_info = matches[matches['match_id'] == MATCH_ID].iloc[0]
print(f"Match sélectionné : {match_info['home_team']} {match_info['home_score']}-{match_info['away_score']} {match_info['away_team']}")

events  = sb.events(match_id=MATCH_ID)
lineups = sb.lineups(match_id=MATCH_ID)
teams   = events['team'].unique().tolist()
TEAM1, TEAM2 = teams[0], teams[1]
print(f"Équipes : {TEAM1} vs {TEAM2}")
print(f"Événements chargés : {len(events):,}")


## 3. 🕸️ Construction du réseau de passes

In [ ]:
def build_pass_network(events, team, period='Full Match', min_passes=3):
    """
    Construit nodes (positions moyennes) et edges (liaisons de passes).
    
    Parameters
    ----------
    events : pd.DataFrame — événements StatsBomb
    team   : str          — nom de l'équipe
    period : str          — 'Full Match', '1st Half', '2nd Half'
    min_passes : int      — seuil minimum de passes par liaison
    
    Returns
    -------
    nodes : pd.DataFrame  — avg_x, avg_y, pass_count par joueur
    edges : pd.DataFrame  — count de passes par paire (player → recipient)
    """
    passes = events[
        (events['type'] == 'Pass') &
        (events['team'] == team) &
        (events['pass_outcome'].isna())   # passes complétées seulement
    ].copy()

    if period == '1st Half':
        passes = passes[passes['period'] == 1]
    elif period == '2nd Half':
        passes = passes[passes['period'] == 2]

    passes['x']     = passes['location'].apply(lambda l: l[0] if isinstance(l, list) else np.nan)
    passes['y']     = passes['location'].apply(lambda l: l[1] if isinstance(l, list) else np.nan)
    passes['end_x'] = passes['pass_end_location'].apply(lambda l: l[0] if isinstance(l, list) else np.nan)
    passes['end_y'] = passes['pass_end_location'].apply(lambda l: l[1] if isinstance(l, list) else np.nan)
    passes = passes.dropna(subset=['x','y','end_x','end_y','pass_recipient'])

    # XI de départ = joueurs ayant passé en 1ère période
    starters = passes[passes['period'] == 1]['player'].value_counts()
    xi = starters[starters >= 1].index.tolist()[:11]

    nodes = (passes[passes['player'].isin(xi)]
             .groupby('player')
             .agg(avg_x=('x','mean'), avg_y=('y','mean'), pass_count=('x','count'))
             .reset_index())

    pair_df = passes[passes['player'].isin(xi) & passes['pass_recipient'].isin(xi)]
    edges   = pair_df.groupby(['player','pass_recipient']).size().reset_index(name='count')
    edges   = edges[edges['count'] >= min_passes]

    return nodes, edges

nodes1, edges1 = build_pass_network(events, TEAM1)
nodes2, edges2 = build_pass_network(events, TEAM2)
print(f"{TEAM1} : {len(nodes1)} nœuds · {len(edges1)} liaisons")
print(f"{TEAM2} : {len(nodes2)} nœuds · {len(edges2)} liaisons")


## 4. 📊 Métriques de centralité (NetworkX)

In [ ]:
def compute_network_metrics(nodes, edges):
    """Ajoute betweenness centrality, in/out degree aux nodes."""
    if nodes.empty or edges.empty:
        return nodes
    G = nx.DiGraph()
    for _, r in nodes.iterrows():
        G.add_node(r['player'])
    for _, r in edges.iterrows():
        G.add_edge(r['player'], r['pass_recipient'], weight=r['count'])

    bc  = nx.betweenness_centrality(G, weight='weight', normalized=True)
    ind = dict(G.in_degree(weight='weight'))
    oud = dict(G.out_degree(weight='weight'))

    nodes = nodes.copy()
    nodes['betweenness'] = nodes['player'].map(bc).fillna(0)
    nodes['in_degree']   = nodes['player'].map(ind).fillna(0)
    nodes['out_degree']  = nodes['player'].map(oud).fillna(0)
    return nodes

nodes1 = compute_network_metrics(nodes1, edges1)
nodes2 = compute_network_metrics(nodes2, edges2)

print(f"\n🏅 Top 5 passeurs — {TEAM1}")
print(nodes1.sort_values('pass_count', ascending=False)[['player','pass_count','betweenness']].head(5).to_string(index=False))
print(f"\n🏅 Top 5 passeurs — {TEAM2}")
print(nodes2.sort_values('pass_count', ascending=False)[['player','pass_count','betweenness']].head(5).to_string(index=False))


In [ ]:
def compute_team_metrics(nodes, edges):
    """Métriques globales : density, clustering, centralization."""
    if nodes.empty or edges.empty:
        return {}
    G = nx.DiGraph()
    for _, r in nodes.iterrows(): G.add_node(r['player'])
    for _, r in edges.iterrows(): G.add_edge(r['player'], r['pass_recipient'], weight=r['count'])

    n           = len(G.nodes)
    density     = nx.density(G) if n > 1 else 0
    clustering  = nx.average_clustering(G.to_undirected()) if n > 1 else 0
    bc          = list(nx.betweenness_centrality(G, normalized=True).values())
    max_bc      = max(bc) if bc else 0
    centralize  = sum(max_bc - v for v in bc) / (n - 1) if n > 1 else 0

    return {
        'density':         round(density, 3),
        'clustering':      round(clustering, 3),
        'centralization':  round(centralize * 100, 1),
        'avg_link_passes': round(edges['count'].mean(), 1),
        'total_passes':    int(nodes['pass_count'].sum()),
        'connections':     len(edges),
    }

m1 = compute_team_metrics(nodes1, edges1)
m2 = compute_team_metrics(nodes2, edges2)

print(f"\n📊 {TEAM1} : {m1}")
print(f"📊 {TEAM2} : {m2}")


## 5. 🎨 Visualisations

In [ ]:
def plot_pass_network(nodes, edges, team_name, period_label='Full Match'):
    pitch = Pitch(pitch_type='statsbomb', pitch_color=DARK_BG,
                  line_color=PITCH_CLR, linewidth=1.5, corner_arcs=True)
    fig, ax = pitch.draw(figsize=(14, 9))
    fig.patch.set_facecolor(DARK_BG)

    if nodes.empty or edges.empty:
        ax.text(60, 40, 'Données insuffisantes', color='white', fontsize=14, ha='center')
        return fig

    max_p = nodes['pass_count'].max()
    max_e = edges['count'].max()

    for _, edge in edges.iterrows():
        fn = nodes[nodes['player'] == edge['player']]
        tn = nodes[nodes['player'] == edge['pass_recipient']]
        if fn.empty or tn.empty: continue
        xs, ys = fn.iloc[0][['avg_x','avg_y']]
        xe, ye = tn.iloc[0][['avg_x','avg_y']]
        alpha = 0.15 + 0.65 * (edge['count'] / max_e)
        lw    = 0.5  + 5.0  * (edge['count'] / max_e)
        ax.annotate('', xy=(xe, ye), xytext=(xs, ys),
                    arrowprops=dict(arrowstyle='->', color=to_rgba(ACCENT, alpha),
                                   lw=lw, connectionstyle='arc3,rad=0.05'))

    for _, node in nodes.iterrows():
        size  = 900 + 2500 * (node['pass_count'] / max_p)
        ax.scatter(node['avg_x'], node['avg_y'], s=size, c=ACCENT,
                   zorder=5, alpha=0.9, linewidths=2, edgecolors='white')
        short = node['player'].split()[-1]
        ax.text(node['avg_x'], node['avg_y'] - 3.8, short,
                color='white', fontsize=8, fontweight='bold',
                ha='center', va='top', zorder=6, fontfamily='monospace')
        ax.text(node['avg_x'], node['avg_y'], str(int(node['pass_count'])),
                color=DARK_BG, fontsize=7.5, fontweight='bold',
                ha='center', va='center', zorder=7)

    ax.set_title(f'{team_name}  |  Pass Network  |  {period_label}',
                 color='white', fontsize=14, fontweight='bold', pad=14)
    plt.tight_layout()
    return fig

# ── Plot les deux réseaux côte à côte
fig, axes = plt.subplots(1, 2, figsize=(22, 9), facecolor=DARK_BG)
plt.subplots_adjust(wspace=0.05)

for ax_idx, (nodes_t, edges_t, team) in enumerate([(nodes1,edges1,TEAM1),(nodes2,edges2,TEAM2)]):
    pitch = Pitch(pitch_type='statsbomb', pitch_color=DARK_BG,
                  line_color=PITCH_CLR, linewidth=1.5, corner_arcs=True)
    pitch.draw(ax=axes[ax_idx])
    axes[ax_idx].set_facecolor(DARK_BG)

    if nodes_t.empty or edges_t.empty:
        continue
    max_p = nodes_t['pass_count'].max()
    max_e = edges_t['count'].max()
    accent = ACCENT if ax_idx == 0 else ACCENT2

    for _, edge in edges_t.iterrows():
        fn = nodes_t[nodes_t['player'] == edge['player']]
        tn = nodes_t[nodes_t['player'] == edge['pass_recipient']]
        if fn.empty or tn.empty: continue
        xs, ys = fn.iloc[0][['avg_x','avg_y']]
        xe, ye = tn.iloc[0][['avg_x','avg_y']]
        alpha  = 0.15 + 0.65 * (edge['count'] / max_e)
        lw     = 0.5  + 5.0  * (edge['count'] / max_e)
        axes[ax_idx].annotate('', xy=(xe,ye), xytext=(xs,ys),
            arrowprops=dict(arrowstyle='->', color=to_rgba(accent, alpha),
                           lw=lw, connectionstyle='arc3,rad=0.05'))

    for _, node in nodes_t.iterrows():
        size = 900 + 2500 * (node['pass_count'] / max_p)
        axes[ax_idx].scatter(node['avg_x'], node['avg_y'], s=size, c=accent,
                    zorder=5, alpha=0.9, linewidths=2, edgecolors='white')
        short = node['player'].split()[-1]
        axes[ax_idx].text(node['avg_x'], node['avg_y']-3.8, short,
            color='white', fontsize=8, fontweight='bold', ha='center', va='top', zorder=6)
        axes[ax_idx].text(node['avg_x'], node['avg_y'], str(int(node['pass_count'])),
            color=DARK_BG, fontsize=7.5, fontweight='bold', ha='center', va='center', zorder=7)

    axes[ax_idx].set_title(team, color='white', fontsize=14, fontweight='bold', pad=10)

fig.suptitle(f'{TEAM1} vs {TEAM2} — Pass Networks', color='white', fontsize=16,
             fontweight='bold', y=1.01)
plt.savefig('pass_networks.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print("✅ Sauvegardé : pass_networks.png")


In [ ]:
# ── Heatmaps de positions de passe
fig, axes = plt.subplots(1, 2, figsize=(22, 9), facecolor=DARK_BG)

for ax_idx, team in enumerate([TEAM1, TEAM2]):
    pitch = Pitch(pitch_type='statsbomb', pitch_color=DARK_BG,
                  line_color=PITCH_CLR, linewidth=1.5)
    pitch.draw(ax=axes[ax_idx])

    team_passes = events[(events['type']=='Pass') & (events['team']==team)].copy()
    team_passes['x'] = team_passes['location'].apply(lambda l: l[0] if isinstance(l,list) else np.nan)
    team_passes['y'] = team_passes['location'].apply(lambda l: l[1] if isinstance(l,list) else np.nan)
    team_passes = team_passes.dropna(subset=['x','y'])

    if not team_passes.empty:
        pitch.kdeplot(team_passes['x'], team_passes['y'], ax=axes[ax_idx],
                     cmap='YlOrRd', fill=True, levels=100, alpha=0.75, bw_adjust=0.7)

    axes[ax_idx].set_title(f'{team} — Passing Heatmap', color='white', fontsize=13, fontweight='bold', pad=10)

fig.suptitle('Passing Heatmaps — Full Match', color='white', fontsize=15, fontweight='bold', y=1.01)
plt.savefig('heatmaps.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print("✅ Sauvegardé : heatmaps.png")


In [ ]:
# ── Centralité : bar charts
fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor=DARK_BG)

for ax_idx, (nodes_t, team, accent) in enumerate([(nodes1,TEAM1,ACCENT),(nodes2,TEAM2,ACCENT2)]):
    ax = axes[ax_idx]
    ax.set_facecolor(DARK_BG)
    if nodes_t.empty or 'betweenness' not in nodes_t.columns:
        continue
    df = nodes_t.sort_values('betweenness', ascending=True).tail(11)
    df['short'] = df['player'].apply(lambda x: x.split()[-1])
    colors = [accent if v == df['betweenness'].max() else '#2d4a5a' for v in df['betweenness']]
    bars = ax.barh(df['short'], df['betweenness'], color=colors, height=0.6)
    for bar, val in zip(bars, df['betweenness']):
        ax.text(bar.get_width()+0.002, bar.get_y()+bar.get_height()/2,
                f'{val:.3f}', va='center', color='#a0aec0', fontsize=9)
    ax.set_xlabel('Betweenness Centrality')
    ax.set_title(f'{team} — Centralité', fontweight='bold', fontsize=12, color='white')
    for spine in ['top','right']: ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig('centrality.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print("✅ Sauvegardé : centrality.png")


## 6. 💾 Export des résultats

In [ ]:
# ── Export nodes + edges en CSV
nodes1['team'] = TEAM1
nodes2['team'] = TEAM2
all_nodes = pd.concat([nodes1, nodes2], ignore_index=True)
all_nodes.to_csv('pass_network_nodes.csv', index=False)

edges1['team'] = TEAM1
edges2['team'] = TEAM2
all_edges = pd.concat([edges1, edges2], ignore_index=True)
all_edges.to_csv('pass_network_edges.csv', index=False)

# ── Export métriques équipe
metrics_df = pd.DataFrame([
    {'team': TEAM1, **m1},
    {'team': TEAM2, **m2},
])
metrics_df.to_csv('team_metrics.csv', index=False)

print("✅ Fichiers exportés :")
print("   pass_network_nodes.csv")
print("   pass_network_edges.csv")
print("   team_metrics.csv")
print()
print(metrics_df.to_string(index=False))
